# **CV #2: Object Detection**

by **R. Ethan Halim** (CS-A 2024, 536718)

⚠️ **The folder of this assignment MUST BE downloaded and viewed locally, in order to be able to display any videos shown in the notebook, as they unfortunately cannot be displayed through GitHub.**

No planning nor writing in this assignment involved any use of LLMs.

## Auxiliary Preface

This assignment primarily uses [numpy](https://numpy.org) for array computations, [numba](https://numba.pydata.org) for performance acceleration, [pygame-ce](https://pyga.me) for text and drawing, and [OpenCV](https://opencv.org) for loading videos. None of these libraries implement the CV algorithms used in this assignment.

Below are the imports to the aforementioned libraries, as well as `IPython`, which is Jupyter-specific to later display the generated videos in the notebook.

In [1]:
import numpy as np
import pygame as pg
import cv2

from IPython.display import display, Video
from numba import njit, prange

pygame-ce 2.5.6 (SDL 2.32.10, Python 3.13.7)


To restate, OpenCV is **solely** used for video loading and saving. No CV-specific functionalities were utilized out of the library. The function below returns the frames of the reference video loaded.

In [2]:
fps: int


def load_video(filename: str) -> list[np.ndarray]:
    cap = cv2.VideoCapture(filename)
    assert cap.isOpened()

    global fps  # For video saving later
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    frames: list[np.ndarray] = []
    
    # Reads and copies all of the frames
    while True:
        success, frame_bgr = cap.read()
        if not success:
            break

        # The rest of the code works with 0.0-1.0 normalized RGB image matrices.
        frame_rgb: np.ndarray = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
        frames.append(frame_rgb.astype(np.float32) / 255.0)

    cap.release()
    display(Video(filename, height=480))
    return frames

The function below encodes and saves a video from an array of frames of the same FPS as the loaded reference video. Both the functions above and below display the video in the notebook for viewing.

In [3]:
def save_video(frames: list[np.ndarray], filename: str):
    height, width, _ = frames[0].shape
    fourcc = cv2.VideoWriter_fourcc(*"avc1")
    out = cv2.VideoWriter(filename, fourcc, fps, (width, height))
    assert out.isOpened()

    for frame_rgb in frames:
        # OpenCV works with 0-255 unnormalized BGR image matrices.
        frame_rgb = (frame_rgb * 255.0).astype(np.uint8)
        out.write(cv2.cvtColor(frame_rgb, cv2.COLOR_RGB2BGR))

    out.release()
    display(Video(filename, height=480))

pygame-ce is used for drawing overlays on the provided images. Specifically, as this assignment deals with image detection, pygame is needed to draw bounding boxes for later demonstrations, as well as rendering text for diagnostic information.

In [4]:
pg.init()
box_font = pg.font.SysFont(None, 32)
caption_font = pg.font.SysFont(None, 48)


def draw(frame: np.ndarray, boxes: list[tuple[pg.Rect, str, str]], caption: str) -> np.ndarray:
    # Converts to pg.Surface from np.ndarray
    surf = pg.surfarray.make_surface((frame.transpose(1, 0, 2) * 255.0).astype(np.uint8))

    for rect, text, color in boxes:
        pg.draw.rect(surf, color, rect, 4)  # Bounding box
        surf.blit(  # Drop shadow
            box_font.render(text, True, "black"),
            (rect.x + 10, rect.y + 10)
        )
        surf.blit(  # Box caption
            box_font.render(text, True, color),
            (rect.x + 8, rect.y + 8)
        )

    surf.blit(caption_font.render(caption, True, "black"), (18, 18))  # Drop shadow
    surf.blit(caption_font.render(caption, True, "white"), (16, 16))  # Caption

    # Converts to np.ndarray from pg.Surface
    return pg.surfarray.pixels3d(surf).transpose(1, 0, 2).astype(np.float32) / 255.0

## Materials

### Footage

Below, I acquired an altered [stock video](https://motionarray.com/stock-video/man-walking-111417) of two men walking across a pavement. The footage is static with a still background, excluding the wavering leaves. The goal is to be able to detect and locate their positions as they walk across the frame.

In [5]:
video = load_video("video.mp4")

### Annotations

In order to be able to later evaluate the performance of the pipeline, I manually annotated the video using [Roboflow](https://roboflow.com) to indicate the ideal bounding boxes of where the human subjects occupy in the screen. I exported the annotations from Roboflow, retrieving the `annotations.coco.json` file, whose content I loaded in the snippet below.

In [6]:
import json  # Built-in module

annotations_json = json.load(open(r"annotations.coco.json"))

# Maps the image IDs according to the JSON to the actual frame number
video_image_ids: dict[int, int] = {}
for image in annotations_json["images"]:
    name = image["file_name"]
    if name.startswith("video-") and name.endswith(".jpg"):
        video_image_ids[image["id"]] = int(name[6:-4]) - 1

# Retrieves the bounding boxes for each frame
annotations: list[list[pg.Rect]] = [[] for _ in video]
for annotation in annotations_json["annotations"]:
    x, y, w, h = map(lambda v: int(float(v)), annotation["bbox"])
    annotations[video_image_ids[annotation["image_id"]]].append(
        pg.Rect((x, y), (w, h))
    )

Below, I visualized the manually-annotated bounding boxes by overlaying them onto the footage.

In [7]:
detection_frames: list[np.ndarray] = []
for mask, annotation in zip(video, annotations):
    detection_frames.append(
        draw(
            mask,
            [(rect, "", "yellow") for rect in annotation],
            f"{len(annotation)} annotation(s)"
        )
    )

save_video(detection_frames, "generated/annotated.mp4")

## Background Modeling

The primary CV technique used to locate the sujects is by extracting the subjects that lie in the foreground through background subtraction by background modeling.

### Mean Image

In order to retrieve the background to subtract the frames by, for each pixel, the mean is taken across the entirety of the video. This technique is employed under the assumption that the foreground would frequently move and wouldn't be "burnt" into the background.

$$B_t(x, y) = \frac{\sum_{f \in \text{frames}} P_f(x, y)}{\text{\# of frames}}$$

The function implementation below implements the direct formula above.

In [8]:
def take_mean(frames: list[np.ndarray]) -> np.ndarray:
    return sum(frames) / len(frames)  # type: ignore

When visualized as a moving-mean as the video progresses, it is as below.

In [9]:
mean_frames: list[np.ndarray] = []
for i in range(len(video)):
    mean_frames.append(take_mean(video[:i+1]))

save_video(mean_frames, "generated/mean.mp4")


However, for consistency, the last frame is taken as the modeled background for the next step. This is essentially the average of all of the frames in the video.

In [10]:
background = mean_frames[-1]
del mean_frames

### Background Subtraction and Thresholding

In order to obtain pixels that differ from the reference background, background subtraction is employed to obtain the pixel value difference.

$$S(x, y) = P(x, y) - B(x, y)$$

However, since only the magnitude of difference is considered, the absolute value is retrieved from the previous formula.

$$\hat{S}(x, y) = |P(x, y) - B(x, y)|$$

In the case below, the foreground is determined to be ones where the difference exceeds a certain threshold.

$$\text{IsForeground}(x, y) \equiv \hat{S}(x, y) \ge \text{threshold}$$

Since each pixel contains an RGB value, a pixel of the "foreground mask" is true when **all** RGB channels exceed the threshold value. It is implemented in the function below.

In [11]:
def threshold_subtraction(frame: np.ndarray, background: np.ndarray, threshold: float) -> np.ndarray:
    difference = np.abs(frame - background)  # Difference image
    return np.all(difference >= threshold, 2)[..., None]  # Mask exceeding the threshold

The mask is visualized below, where **black** pixels are part of the background, and **white** pixels are part of the foreground.

In [12]:
masks: list[np.ndarray] = []
mask_frames: list[np.ndarray] = []

for mask in video:
    # The threshold of difference is set to 20%.
    mask = threshold_subtraction(mask, background, 0.2)
    masks.append(mask)
    mask_frames.append(
        # Maps mask to an RGB image
        np.where(
            mask,
            np.array((1.0, 1.0, 1.0), dtype=np.float32),
            np.array((0.0, 0.0, 0.0), dtype=np.float32)
        )
    )

save_video(mask_frames, "generated/mask.mp4")
del mask_frames

### Mode (Median) Filter

A median filter typically sweeps over the image, taking the median value of each pixel's neighbors to assign it as its own. If the kernel is a square encircling a pixel with radius $k$, it is mathematically expressed like below.

$$
M(x, y) = \text{Median}([p \text{ | } P(dx, dy) \text{, where }
\begin{cases}
x - k \le dx \le x + k \\
y - k \le dy \le y + k
\end{cases}
])
$$

However, I intended to use it to clean the stray pixel noise in the foreground mask. As the pixels of an image mask are booleans, either true or false, taking the mode (majority values) is equivalent to taking the median. Hence, that is what the function implementation below does.

I used numba to expedite the function's execution, since convolution is an expensive process to be done in pure Python synchronously. The `@njit` function decorator and `prange` allows for the function to be executed in parallel for every pixel.

In [13]:
@njit
def filter_mode(mask: np.ndarray, radius: int) -> np.ndarray:
    filtered = np.empty(mask.shape, np.bool)
    kernel_size = radius * 2 + 1
    kernel_area = kernel_size * kernel_size

    # Kernel sweep for each pixel in parallel
    for y in prange(mask.shape[0]):
        for x in prange(mask.shape[1]):
            ones = 0

            # Checks itself and its neighbors
            for dy in range(y - radius, y + radius + 1):
                # Skips if it is outside of the image's frame 
                if not (0 <= dy < mask.shape[0]):
                    continue

                for dx in range(x - radius, x + radius + 1):
                    # Counts the ones
                    if 0 <= dx < mask.shape[1] and mask[dy][dx][0]:
                        ones += 1

            if ones > kernel_area // 2:
                # The majority (mode) is 1.
                filtered[y][x][0] = True
            else:
                # The majority (mode) is 0.
                filtered[y][x][0] = False

    return filtered

Below is the denoised mask following the filtering.

In [14]:
filtered_masks: list[np.ndarray] = []
filtered_mask_frames: list[np.ndarray] = []

for mask in masks:
    filtered_mask = filter_mode(mask, 2)
    filtered_masks.append(filtered_mask)
    filtered_mask_frames.append(
        # Maps mask to an RGB image
        np.where(
            filtered_mask,
            np.array((1.0, 1.0, 1.0), dtype=np.float32),
            np.array((0.0, 0.0, 0.0), dtype=np.float32)
        )
    )

save_video(filtered_mask_frames, "generated/filtered.mp4")
del filtered_mask_frames

### Foreground Separation

When visualized alternatively, the mask can be applied to the original video frames to display the supposed elements of the foreground.

$$
F(x, y) = \begin{cases}
\text{If } M(x, y) = 1, & P(x, y) \\
\text{Otherwise,} & \text{black}
\end{cases}
$$

It is visualized as below.

In [15]:
fg_frames: list[np.ndarray] = []
for frame, mask in zip(video, filtered_masks):
    fg_frames.append(frame * mask)

save_video(fg_frames, "generated/foreground.mp4")

The arms are unfortunately not masked due to the similarity in color with the background. Reducing the threshold following background subtraction added more noise. Thus, this compromise had to be taken.

## Connected-Component Labeling

In order to distinguish objects from the given foreground, an object is defined as a connected "blob" in the foreground mask. This necessitates connected-component labeling, which requires a union-find (disjoint set) data structure. Below, the find and union operations were implemented under numba, where nodes are stored as a numpy array.

In [16]:
@njit
def set_find(parents: np.ndarray, x: int):
    while parents[x] != x:
        # Path compression
        parents[x] = parents[parents[x]]
        x = parents[x]

    return x


@njit
def set_union(parents: np.ndarray, a: int, b: int):
    ra = set_find(parents, a)
    rb = set_find(parents, b)

    if ra != rb:
        if ra < rb:
            parents[rb] = ra
        else:
            parents[ra] = rb

The implementation of a four-neighbor connected-component labeling function is done below. Algorithmic explanations are provided in the comments.

In [17]:
@njit
def label_connected(mask: np.ndarray) -> np.ndarray:
    h, w, _ = mask.shape
    out = np.zeros((h, w, 1), dtype=np.uint32)

    # Disjoint set array to be used find and union operations on
    parents = np.empty(h * w + 1, dtype=np.uint32)
    parents[0] = 0

    # First pass propagating downleft
    next_label = 1
    for i in range(h):
        for j in range(w):
            if not mask[i, j, 0]:
                continue

            # Out-of-bounds check
            up = out[i - 1, j, 0] if i > 0 else 0
            left = out[i, j - 1, 0] if j > 0 else 0

            # Checks four-neighbor connectiveness
            if up == 0 and left == 0:
                out[i, j, 0] = next_label
                parents[next_label] = next_label
                next_label += 1
            elif up != 0 and left == 0:
                out[i, j, 0] = up
            elif up == 0 and left != 0:
                out[i, j, 0] = left
            else:
                out[i, j, 0] = up
                set_union(parents, up, left)  # Unites when connected

    # Second pass to compress labels and flatten the ID image
    root_to_new = np.zeros(next_label, dtype=np.uint32)
    new_id = 1
    for i in range(h):
        for j in range(w):
            lab = out[i, j, 0]
            if lab == 0:  # Background
                continue

            root = set_find(parents, lab)  # Internally does path compression
            if root_to_new[root] == 0:
                root_to_new[root] = new_id
                new_id += 1
            out[i, j, 0] = root_to_new[root]

    # Following the second pass, each pixel would be an integer at most equivalent to the number
    # of labels. A background pixel is set to zero.
    return out

Below is a visualization of the different "objects" that the it recognizes each frame. A whole "object" is marked by a single shade. Patches that are disconnected are indicated by their differering shades.

In [18]:
components: list[np.ndarray] = []
component_frames: list[np.ndarray] = []

for mask in filtered_masks:
    component_frame = label_connected(mask)
    components.append(component_frame)
    component_frames.append(
        # Maps mask to an RGB image
        np.ones((*component_frame.shape[:2], 3), np.float32)
        * component_frame
        / max(1, np.max(component_frame))
    )

save_video(component_frames, "generated/connected.mp4")
del component_frames

## Bounding Boxes

The techniques below weren't mentioned in the materials. However, they chiefly operate to turn the foreground "blobs" into bounding boxes for the purpose of scoring in the end.

### Box-Bounding

For each blob, the most left, right, top, and bottom values are taken note of to obtain a bounding box fitting the blob. It is implemented in the functions below. `_bound_boxes` was separated and optimized using numba, as numba cannot compile `bound_boxes` directly due to the use of the `pg.Rect` type (representing the bounding boxes), which does not belong to numpy.

Additionally, `min_area` filters out bounding boxes whose area does not exceed the threshold specified, in order to get rid of miniscule disjoint pixels. Algorithmic explanations are provided in the comments.

In [19]:
@njit
def _bound_boxes(component_areas: np.ndarray, component_bounds: np.ndarray, components: np.ndarray):
    # For every pixel
    for y in range(components.shape[0]):
        for x in range(components.shape[1]):
            component = components[y][x][0]
            if component == 0:  # Skips background pixels
                continue

            # Box ID-ing starts at 0.
            component -= 1
            component_areas[component] += 1  # Increases the area by one pixel

            # If a pixel is inside of the blob, but outside of the known bounding box,
            # the size of the bounding box is increased to accommodate.
            ax, ay, bx, by = component_bounds[component]
            ax = x if x < ax else ax
            ay = y if y < ay else ay
            bx = x if x > bx else bx
            by = y if y > by else by
            component_bounds[component] = np.array((ax, ay, bx, by), np.uint32)


def bound_boxes(components: np.ndarray, min_area: int) -> list[pg.Rect]:
    component_count = np.max(components)  # The number of components is the maximum ID value
    component_areas = np.zeros((component_count,), np.uint32)  # Stores the area of each component
    component_bounds = np.tile(
        # Ideally, each box should span from (-inf, -inf) to (inf, inf). This is done so that the
        # bound enlargement code is overridden on pixels of IDs that it has never encountered. 
        np.array((components.shape[1], components.shape[0], 0, 0), np.uint32),
        (component_count, 1)
    )

    _bound_boxes(component_areas, component_bounds, components)

    boxes: list[pg.Rect] = []
    for component in range(component_count):
        # Filters components that are smaller than the minimum area
        if component_areas[component] >= min_area:
            ax, ay, bx, by = component_bounds[component]
            boxes.append(pg.Rect((ax, ay, bx - ax, by - ay)))

    return boxes

The bounding boxes obtained from the foreground mask of each frame is visualized below.

In [20]:
# The bounding box area threshold is set to be 10,000 pixels.
bboxes = [bound_boxes(component, 10_000) for component in components]
bbox_frames: list[np.ndarray] = []

for frame, bbox in zip(video, bboxes):
    bbox_frames.append(
        # Visualizes the bounding boxes
        draw(
            frame,
            [(rect, "", "yellow") for rect in bbox],
            f"{len(bbox)} box(es)"
        )
    )

save_video(bbox_frames, "generated/bbox.mp4")
del bbox_frames

### Merging Overlaps

In frames around 0:03, it can be seen that two bounding boxes were overlayed on the same object while overlapping. This issue can be alleviated by merging overlapping bounding boxes into one by taking a rectangle of the least area to fit both.

The function below implements the logic in $O(n^2)$ for all bounding boxes against every other bounding box. Algorithmic explanations are provided in the comments.

In [21]:
def merge_overlaps(bboxes: list[pg.Rect]) -> list[pg.Rect]:
    # For every bounding box
    for i, bbox in enumerate(bboxes):
        if bbox is None:  # If merged with something else already
            continue

        # For every other bounding box following it
        for j in range(i + 1, len(bboxes)):
            other_bbox = bboxes[j]
            if other_bbox is not None and bbox.colliderect(bboxes[j]):
                bbox = bbox.union(bboxes[j])  # Takes the rectangle union when intersecting
                bboxes[j] = None  # type: ignore # Removes the other

        bboxes[i] = bbox

    # Removes the removed bounding boxes that were overwritten as `None`
    return list(filter(lambda x: x is not None, bboxes))

The cleaned bounding boxes following overlap-merging were obtained and are visualized below.

In [22]:
detections: list[list[pg.Rect]] = []
detection_frames: list[np.ndarray] = []

for frame, bbox in zip(video, bboxes):
    detection = merge_overlaps(bbox)
    detections.append(detection)
    detection_frames.append(
        # Visualizes the bounding boxes
        draw(
            frame,
            [(rect, "", "yellow") for rect in detection],
            f"{len(detection)} detection(s)"
        )
    )

save_video(detection_frames, "generated/detected.mp4")
del detection_frames

## Scoring

### Intersection over Union

The correctness of a predicted bounding box is dictated by the area of intersection divided the area of united areas of both the prediction and ground truth.

$$
a_0 = \frac{\text{area}(\text{BB}_\text{p} \cap \text{BB}_\text{gt})}
{\text{area}(\text{BB}_\text{p} \cup \text{BB}_\text{gt})}
$$

As there are multiple ground truths, each prediction is compared to each possible ground truth, whose IoU value is taken and taken the overall maximum of.

$$
a_0 = \text{max}(\{\frac{\text{area}(\text{BB}_\text{p} \cap \text{BB}_\text{gt})}
{\text{area}(\text{BB}_\text{p} \cup \text{BB}_\text{gt})} \text{ | }
\text{BB}_\text{gt} \in \text{ground truths}\})
$$

It is implemented as the function below.

In [23]:
def get_iou(bbox: pg.Rect, reference_bboxes: list[pg.Rect]) -> tuple[pg.Rect | None, float]:
    area = bbox.width * bbox.height

    max_iou = 0.0
    rect = None  # Keeps track of which ground truth it matches
    for reference in reference_bboxes:
        # Ignores ground truths that don't intersect
        if not bbox.colliderect(reference):
            continue

        clipping = bbox.clip(reference)  # Gets the rectangle of the intersection
        intersection = clipping.width * clipping.height
        union = area + reference.width * reference.height - intersection

        # Obtains the IoU and overtakes the maximum if higher
        iou = intersection / union
        if iou > max_iou:
            rect = reference
            max_iou = iou

    return rect, max_iou

### Accuracy

Using an IoU threshold of 50%, a predicted bounding box is deemed to be correct if its IoU exceeds that threshold, which is registered as a TP (true positive). Incorrect ones are labeled as FP (false positives). Ground truths which were not detected at all are labeled as FN (false negatives).

Using the three values of a binary confusion matrix, values such as accuracy, and precision, recall can be calculated.

$$
\begin{cases}
\text{Accuracy} &= \frac{TP}{\text{\# of ground truths}} \\
\text{Precision} &= \frac{TP}{TP + FP} \\
\text{Recall} &= \frac{TP}{TP + FN}
\end{cases}
$$

The scoring code snippet below annotates the video with the information for each bounding box and overall scores on a per-frame basis.

In [24]:
# Keeps track of TP, FP, FN, and total ground truth annotations
total_tp = total_fp = total_fn = total_annotations = 0

scoring_frames: list[np.ndarray] = []
for frame, detection, annotation in zip(video, detections, annotations):
    detected_annotations: list[pg.Rect] = []
    true_detections: dict[tuple[int, int, int, int], float] = {}  # Maps to IoU
    false_detections: dict[tuple[int, int, int, int], float] = {}  # Maps to IoU

    # For each bounding ox predicted
    for bbox in detection:
        target, iou = get_iou(bbox, annotation)
        if target is not None and iou >= 0.50:  # The IoU threshold is set to be 50%.
            # Correctly detected and located
            detected_annotations.append(target)
            true_detections[(bbox.x, bbox.y, bbox.w, bbox.h)] = iou
        else:
            # Incorrectly detected nor located
            false_detections[(bbox.x, bbox.y, bbox.w, bbox.h)] = iou

    # Aggregates the TP, FP, FN, and total ground truth annotations
    total_tp += (tp := len(detected_annotations))
    total_fp += (fp := len(false_detections))
    total_fn += (fn := len(annotation) - len(detected_annotations))
    total_annotations += len(annotation)

    # Calculates the accuracy, precision, and recall for the frame with zero-division protection
    accuracy = tp / len(annotation) if len(annotation) > 0 else 0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0

    scoring_frames.append(
        draw(
            frame,

            # Undetected ground truth annotations are in gray.
            [(rect, "", "gray") for rect in annotation]

            # Detected ground truth annotations are in yellow.
            + [(rect, "", "yellow") for rect in detected_annotations]

            # Correctly predicted bounding boxes are in green.
            + [(pg.Rect(rect), f"{iou * 100:.2f}%", "green") for rect, iou in true_detections.items()]

            # Incorrectly predicted bounding boxes are in red.
            + [(pg.Rect(rect), f"{iou * 100:.2f}%", "red") for rect, iou in false_detections.items()],

            # Diagnostics
            f"TP: {tp}   FP: {fp}   FN: {fn}\n"
            f"Accuracy: {accuracy:.4f}\n"
            f"Precision: {precision:.4f}\n"
            f"Recall: {recall:.4f}"
        )
    )

save_video(scoring_frames, "generated/scoring.mp4")
del scoring_frames

Using the cumulative confusion matrix values, I obtain the accuracy, precision, and recall value of the pipeline over the entire video using the same formulae.

In [25]:
print(
    f"Accuracy\t: {total_tp / total_annotations * 100:.4f}%\n"
    f"Precision\t: {total_tp / (total_tp + total_fp) * 100:.4f}%\n"
    f"Recall\t\t: {total_tp / (total_tp + total_fn) * 100:.4f}%"
)

Accuracy	: 62.1469%
Precision	: 61.1111%
Recall		: 62.1469%


We can conclude the using the techniques described and implemented in this notebook, the pipeline may achieve an overall ~60% accuracy to detect and locate bypassers on a non-dense street with a static background and camera angle.